# Create and Load Clinical Trial Evidence Table in BigQuery

This notebook:
1. Authenticates to BigQuery with a service account JSON file
2. Executes a SQL script to create the destination table
3. Loads CSV data into BigQuery
4. Validates the loaded row count

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from pathlib import Path
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# Resolve unmet_needs root even when running from ddl_scripts
NOTEBOOK_DIR = Path.cwd()
UNMET_NEEDS_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "ddl_scripts" else NOTEBOOK_DIR

SQL_FILE_PATH = UNMET_NEEDS_DIR / "ddl_scripts" / "sql_scripts" / "create_clinical_trial_evidence_table.sql"
CSV_FILE_PATH = Path(r"C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\notebooks\unmet_needs_to_trials_mapping_updated.csv")

# Update this path to your service account key file if not using env var
SERVICE_ACCOUNT_JSON_PATH = Path(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS", str(UNMET_NEEDS_DIR / "service_account.json"))
)

print(f"Unmet needs dir: {UNMET_NEEDS_DIR}")
print(f"SQL path exists: {SQL_FILE_PATH.exists()} -> {SQL_FILE_PATH}")
print(f"CSV path exists: {CSV_FILE_PATH.exists()} -> {CSV_FILE_PATH}")
print(f"Service JSON exists: {SERVICE_ACCOUNT_JSON_PATH.exists()} -> {SERVICE_ACCOUNT_JSON_PATH}")

Unmet needs dir: c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs
SQL path exists: True -> c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\ddl_scripts\sql_scripts\create_clinical_trial_evidence_table.sql
CSV path exists: True -> C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\notebooks\unmet_needs_to_trials_mapping_updated.csv
Service JSON exists: True -> C:\Users\p00102977\Desktop\Projects\API\repo\cognito-prod-394707-d38a0283cb16.json


In [3]:
# Metadata sourced from ddl_scripts/clinical_trial_evidence_to_unmet_needs_mapping.json
PROJECT_ID = "cognito-prod-394707"
DATASET_ID = "cognito_prod_datamart"
TABLE_NAME = "clinical_trial_evidence_to_unmet_need_mapping"


TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

SCHEMA = [
    bigquery.SchemaField("trial_id", "STRING"),
    bigquery.SchemaField("molecule_name", "STRING"),
    bigquery.SchemaField("unmet_need", "STRING"),
    bigquery.SchemaField("category", "STRING"),
    bigquery.SchemaField("country", "STRING"),
    bigquery.SchemaField("addresses_unmet_need", "STRING"),
    bigquery.SchemaField("evidence_strength", "STRING"),
    bigquery.SchemaField("relevant_endpoint", "STRING"),
    bigquery.SchemaField("explanation", "STRING"),
    bigquery.SchemaField("search_sources_used", "STRING"),
    bigquery.SchemaField("phase", "STRING"),
    bigquery.SchemaField("phase_status", "STRING"),
    bigquery.SchemaField("trial_size", "FLOAT64"),
    bigquery.SchemaField("trial_location", "STRING"),
]

print(f"Target table: {TABLE_ID}")

Target table: cognito-prod-394707.cognito_prod_datamart.clinical_trial_evidence_to_unmet_need_mapping


In [4]:
if not SERVICE_ACCOUNT_JSON_PATH.exists():
    raise FileNotFoundError(
        f"Service account JSON not found: {SERVICE_ACCOUNT_JSON_PATH}. Set GOOGLE_APPLICATION_CREDENTIALS or update SERVICE_ACCOUNT_JSON_PATH."
    )

credentials = service_account.Credentials.from_service_account_file(str(SERVICE_ACCOUNT_JSON_PATH))
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"Authenticated BigQuery client for project: {client.project}")

Authenticated BigQuery client for project: cognito-prod-394707


In [5]:
if not SQL_FILE_PATH.exists():
    raise FileNotFoundError(f"SQL script not found: {SQL_FILE_PATH}")

ddl_sql = SQL_FILE_PATH.read_text(encoding="utf-8").strip()
if not ddl_sql:
    raise ValueError("SQL script is empty.")

ddl_job = client.query(ddl_sql)
ddl_job.result()
print(f"Table creation DDL executed successfully. Job ID: {ddl_job.job_id}")

Table creation DDL executed successfully. Job ID: 7c0c3d2f-11eb-431c-93c5-174b18bc9cc3


In [16]:
if not CSV_FILE_PATH.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_FILE_PATH}")

df = pd.read_csv(CSV_FILE_PATH)

required_cols = [field.name for field in SCHEMA]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"CSV missing required columns: {missing_cols}")

# Keep only table columns in schema order
df = df[required_cols].copy()

# Coerce all STRING fields to string so pyarrow does not infer incompatible dtypes (e.g., bool)
string_cols = [field.name for field in SCHEMA if field.field_type == "STRING"]
for col in string_cols:
    df[col] = df[col].astype("string")
    df[col] = df[col].where(df[col].notna(), None)

# Normalize trial_size to numeric for FLOAT64 load
df["trial_size"] = pd.to_numeric(df["trial_size"], errors="coerce")

print(f"CSV loaded with {len(df)} rows and {len(df.columns)} columns.")
print(f"addresses_unmet_need dtype after coercion: {df['addresses_unmet_need'].dtype}")
df.head(3)

CSV loaded with 67 rows and 14 columns.
addresses_unmet_need dtype after coercion: string


,trial_id,molecule_name,unmet_need,category,country,addresses_unmet_need,evidence_strength,relevant_endpoint,explanation,search_sources_used,phase,phase_status,trial_size,trial_location
0,EUCTR2023-503657-35 (TRIUMPH-1),Retatrutide,Maintaining weight loss over the long term rem...,Durability of Disease Control,US,True,Strong,Percent Change from Baseline in Body Weight,"The Phase 3 TRIUMPH-1 trial, evaluating retatr...",['https://www.retatrutide.med/clinical-trials/...,3,Active,2100.0,EU
1,EUCTR2023-503657-35 (TRIUMPH-1),Retatrutide,Obstructive sleep apnea and sleep-disordered b...,Complication Prevention & Management,US,True,Moderate,Change in Apnea-Hypopnea Index (AHI),"The TRIUMPH clinical development program, whic...",['https://vertexaisearch.cloud.google.com/grou...,3,Active,2100.0,EU
2,EUCTR2023-503657-35 (TRIUMPH-1),Retatrutide,Diabetic patients experience an impaired quali...,Quality of Life & Psychosocial Burden,France,True,Moderate,Reduction in knee pain,The TRIUMPH-1 trial's design includes protocol...,['https://vertexaisearch.cloud.google.com/grou...,3,Active,2100.0,EU


In [17]:
job_config = bigquery.LoadJobConfig(
    schema=SCHEMA,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_dataframe(df, TABLE_ID, job_config=job_config)
load_job.result()
print(f"Data load completed. Job ID: {load_job.job_id}")
print(f"Rows written: {load_job.output_rows}")

Data load completed. Job ID: 6df9b2a1-4414-416d-9639-3314e5f50fd4
Rows written: 67


In [ ]:
validation_query = f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT molecule_name) AS molecule_count
FROM `{TABLE_ID}`
"""

result_df = client.query(validation_query).result().to_dataframe()
print("Validation summary:")
print(result_df.to_string(index=False))